Họ và tên: Trần Hữu Thịnh

MSSV: 23521512

Lớp: DS200.M21

Predict YouTube Trending Videos - 2026

Dataset: [Trending YouTube Videos 113 Countries](https://www.kaggle.com/datasets/asaniczka/trending-youtube-videos-113-countries)

# **Predict YouTube Trending Videos**

## **Import Libraries**

In [ ]:
!pip install -q kagglehub pyspark==3.3.0

In [ ]:
import numpy as np
import pandas as pd
import time
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import os
import glob

from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, accuracy_score
from IPython.core.display import display
from pyspark.sql import SparkSession
from pyspark.sql.types import FloatType
from pyspark.sql import functions as f
from pyspark.sql import Row
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, NaiveBayes
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import MinMaxScaler, StandardScaler
from pyspark.ml.pipeline import PipelineModel

sns.set()
pd.set_option('display.max_columns', None)

# Initialize Spark Session
spark = SparkSession.builder.appName('YouTubeTrending').config("spark.executor.memory","16g").getOrCreate()
print("✓ Spark Session created successfully!")
spark

## **Download YouTube Trending Dataset**

In [ ]:
# Download dataset from KaggleHub
print("Downloading YouTube Trending Videos dataset...")
path = kagglehub.dataset_download("asaniczka/trending-youtube-videos-113-countries")
print(f"✓ Dataset downloaded at: {path}")

# List available files
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
print(f"\nFound {len(csv_files)} CSV files")
for i, f in enumerate(csv_files[:10]):
    print(f"  {i+1}. {os.path.basename(f)}")
if len(csv_files) > 10:
    print(f"  ... and {len(csv_files)-10} more files")

## **Read and Prepare Dataset**

In [ ]:
# Load and combine CSV files
df_list = []
for csv_file in csv_files[:5]:  # Load first 5 countries for demo
    try:
        df_temp = pd.read_csv(csv_file)
        df_list.append(df_temp)
        print(f"✓ Loaded: {os.path.basename(csv_file)} - Shape: {df_temp.shape}")
    except Exception as e:
        print(f"✗ Error loading {os.path.basename(csv_file)}: {e}")

# Combine all dataframes
df_combined = pd.concat(df_list, ignore_index=True)
print(f"\n✓ Combined shape: {df_combined.shape}")
print(f"\nColumns: {list(df_combined.columns)}")

In [ ]:
# Display first rows
print("Dataset Info:")
df_combined.info()
print("\n")
df_combined.head()

In [ ]:
# Data Cleaning
print("Data Cleaning...")

# Remove duplicates
df_combined = df_combined.drop_duplicates()

# Fill missing values
numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_combined[col].isnull().sum() > 0:
        df_combined[col].fillna(df_combined[col].median(), inplace=True)

categorical_cols = df_combined.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_combined[col].isnull().sum() > 0:
        df_combined[col].fillna(df_combined[col].mode()[0], inplace=True)

print(f"✓ Shape after cleaning: {df_combined.shape}")
print(f"✓ Missing values: {df_combined.isnull().sum().sum()}")

In [ ]:
# Create PySpark DataFrame
print("Converting to PySpark DataFrame...")
dataset = spark.createDataFrame(df_combined)
dataset.printSchema()
print(f"\nTotal records: {dataset.count()}")
dataset.show(10, False)

## **Feature Engineering**

In [ ]:
# Select and rename important columns
data = dataset.select(
    f.col('title'),
    f.col('channelTitle'),
    f.col('categoryId'),
    f.col('view_count').cast('long').alias('view_count'),
    f.col('likes').cast('long').alias('likes'),
    f.col('comment_count').cast('long').alias('comment_count'),
    f.col('tag_count').cast('int').alias('tag_count'),
    f.col('description_length').cast('int').alias('description_length')
)

# Remove null values
data = data.dropna()

print(f"Data shape after feature selection: {data.count()} records")
data.printSchema()
data.show(10, False)

In [ ]:
# Create label: TRENDING (1 if view_count > median, else 0)
median_views = data.approxQuantile('view_count', [0.5], 0.01)[0]
print(f"Median views: {median_views}")

data = data.withColumn(
    'LABEL',
    f.when(f.col('view_count') > median_views, 1).otherwise(0)
)

# Create feature columns
data = data.withColumn('like_ratio', f.col('likes') / (f.col('view_count') + 1))
data = data.withColumn('comment_ratio', f.col('comment_count') / (f.col('view_count') + 1))
data = data.withColumn('engagement', f.col('likes') + f.col('comment_count'))

# Select final features
data = data.select(
    'title', 'channelTitle', 'categoryId', 'view_count', 'likes',
    'comment_count', 'tag_count', 'description_length',
    'like_ratio', 'comment_ratio', 'engagement', 'LABEL'
)

print(f"\n✓ Features created!")
data.show(10, False)

In [ ]:
# Check label distribution
print("Label Distribution:")
data.groupBy('LABEL').count().show()

## **Prepare Machine Learning Pipeline**

In [ ]:
# Select features for modeling
feature_cols = ['tag_count', 'description_length', 'like_ratio', 'comment_ratio', 'engagement']

# Create feature vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features'
)

# Normalize features
scaler = MinMaxScaler(inputCol='features', outputCol='scaledFeatures')

# Create feature vector
data_vectorized = assembler.transform(data)
data_scaled = scaler.fit(data_vectorized).transform(data_vectorized)

# Select train data
train_data = data_scaled.select('scaledFeatures', 'LABEL')
train_data.show(10, False)

In [ ]:
# Split data: 70% train, 30% test
train, test = train_data.randomSplit([0.7, 0.3], seed=42)
print(f"Training set size: {train.count()}")
print(f"Test set size: {test.count()}")

## **Train Classification Models**

In [ ]:
# 1. Logistic Regression
print("\n" + "="*60)
print("1. LOGISTIC REGRESSION")
print("="*60)

lr = LogisticRegression(featuresCol='scaledFeatures', labelCol='LABEL', maxIter=100)
lr_model = lr.fit(train)

# Predictions
lr_pred = lr_model.transform(test)

# Evaluation
evaluator = MulticlassClassificationEvaluator(
    predictionCol='prediction',
    labelCol='LABEL',
    metricName='accuracy'
)
lr_accuracy = evaluator.evaluate(lr_pred)
print(f"Accuracy: {lr_accuracy:.4f}")

# F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(
    predictionCol='prediction',
    labelCol='LABEL',
    metricName='f1'
)
lr_f1 = evaluator_f1.evaluate(lr_pred)
print(f"F1 Score: {lr_f1:.4f}")

In [ ]:
# 2. Decision Tree
print("\n" + "="*60)
print("2. DECISION TREE")
print("="*60)

dt = DecisionTreeClassifier(featuresCol='scaledFeatures', labelCol='LABEL', maxDepth=10)
dt_model = dt.fit(train)

dt_pred = dt_model.transform(test)
dt_accuracy = evaluator.evaluate(dt_pred)
dt_f1 = evaluator_f1.evaluate(dt_pred)

print(f"Accuracy: {dt_accuracy:.4f}")
print(f"F1 Score: {dt_f1:.4f}")

In [ ]:
# 3. Random Forest
print("\n" + "="*60)
print("3. RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    featuresCol='scaledFeatures',
    labelCol='LABEL',
    numTrees=50,
    maxDepth=10
)
rf_model = rf.fit(train)

rf_pred = rf_model.transform(test)
rf_accuracy = evaluator.evaluate(rf_pred)
rf_f1 = evaluator_f1.evaluate(rf_pred)

print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1 Score: {rf_f1:.4f}")

In [ ]:
# 4. Naive Bayes
print("\n" + "="*60)
print("4. NAIVE BAYES")
print("="*60)

nb = NaiveBayes(featuresCol='scaledFeatures', labelCol='LABEL')
nb_model = nb.fit(train)

nb_pred = nb_model.transform(test)
nb_accuracy = evaluator.evaluate(nb_pred)
nb_f1 = evaluator_f1.evaluate(nb_pred)

print(f"Accuracy: {nb_accuracy:.4f}")
print(f"F1 Score: {nb_f1:.4f}")

## **Model Comparison**

In [ ]:
# Compare all models
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Naive Bayes'],
    'Accuracy': [lr_accuracy, dt_accuracy, rf_accuracy, nb_accuracy],
    'F1 Score': [lr_f1, dt_f1, rf_f1, nb_f1]
})

print(results.to_string(index=False))

# Find best model
best_model_idx = results['Accuracy'].idxmax()
best_model = results.iloc[best_model_idx]
print(f"\n✓ Best Model: {best_model['Model']} (Accuracy: {best_model['Accuracy']:.4f})")

In [ ]:
# Save best model
import os
model_path = '/tmp/youtube_trending_model'
if best_model_idx == 0:
    lr_model.save(model_path)
    print(f"✓ Logistic Regression model saved to {model_path}")
elif best_model_idx == 1:
    dt_model.save(model_path)
    print(f"✓ Decision Tree model saved to {model_path}")
elif best_model_idx == 2:
    rf_model.save(model_path)
    print(f"✓ Random Forest model saved to {model_path}")
else:
    nb_model.save(model_path)
    print(f"✓ Naive Bayes model saved to {model_path}")

## **Make Predictions**

In [ ]:
# Show predictions on test set
print("Sample Predictions (Best Model):")
if best_model_idx == 0:
    predictions = lr_pred
elif best_model_idx == 1:
    predictions = dt_pred
elif best_model_idx == 2:
    predictions = rf_pred
else:
    predictions = nb_pred

predictions.select('scaledFeatures', 'LABEL', 'prediction', 'probability').show(10, False)

In [ ]:
# Detailed classification report
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("\n" + "="*60)
print("DETAILED METRICS (Best Model)")
print("="*60)

# Convert to pandas for detailed metrics
predictions_pd = predictions.toPandas()
print(classification_report(predictions_pd['LABEL'], predictions_pd['prediction']))